In [1]:
from KVerse_vector_space.vector_main import VectorStore

vs = VectorStore(index_name = 'kverse-books')

/Users/sukesh/Desktop/kverse/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:


query_text = f"""
    SELECT *
    FROM c
"""
all_results = vs.index.container.query_items(
    query=query_text,
    enable_cross_partition_query=True
        )

In [12]:
from KVerse_database_system.db_utils import retrieve_document, update_document
for result in all_results:
    result = result['metadata']
    if result['Granularity'] == 'sub_chapter':
        id = result['id']
        subject_code = result['Subject_Code']
        chapter_name = result['Chapter_Index']
        sub_chapter_name = result['Sub_Chapter_Index']
        subject_document = retrieve_document('Subject_Document', primary_key=subject_code)
        subject_document = subject_document[0]
        chapters = subject_document.get('Chapters', {})
        if chapter_name not in chapters:
            chapters[chapter_name] = {}
        if sub_chapter_name not in chapters[chapter_name]:
            chapters[chapter_name][sub_chapter_name] = []
        chapters[chapter_name][sub_chapter_name].append(id) 
        update_document(
            collection_name='Subject_Document',
            query={'Subject_Code': subject_code},
            updates={'Chapters': chapters})